# Two-stage detector: Faster R-CNN on VisDrone

**To run:** Runtime → Change runtime type → GPU, check the paths in the settings cell, then Runtime → Run all.

Training saves to Drive after every epoch. If Colab disconnects, run everything again and it continues from the last saved epoch.

## 1. Setup

In [ ]:
!pip install -q pycocotools

import shutil, time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
import yaml
from google.colab import drive
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU - turn on the GPU runtime')

In [ ]:
# ---------- Settings: this is the only cell you should need to edit ----------
DRIVE_ROOT = Path('/content/drive/MyDrive/hw2_train-s3_val-s3')  # dataset folder on Drive
ROOT = Path('/content/hw2_train-s3_val-s3')                      # fast local copy
OUT = Path('/content/drive/MyDrive/hw2_runs')                    # checkpoints and results

EPOCHS = 6              # final model
EXP_EPOCHS = 2          # each parameter experiment
EXP_TRAIN_IMAGES = 2000 # experiments use a part of the data, so they finish in reasonable time
EXP_VAL_IMAGES = 500

BATCH_SIZE = 2          # lower MIN_SIZE to 800 if you get CUDA out of memory
LR = 0.005
MIN_SIZE, MAX_SIZE = 1024, 1824
MAX_DET = 300           # scenes are dense
SCORE_THRESH = 0.01     # low threshold is standard for mAP

drive.mount('/content/drive')
OUT.mkdir(parents=True, exist_ok=True)
if not ROOT.exists():
    print('Copying dataset to local disk...')
    shutil.copytree(DRIVE_ROOT, ROOT)

## 2. Data

Boxes come from `annotations_coco.json`. The dataset class handles the cases that caused problems:

- Frames listed in the json but missing from disk (removed by preprocessing) are skipped, and they are left out of evaluation too.
- Annotations whose category isn't listed in the json are dropped, so a label can never be out of range for the model.
- Zero-size boxes are dropped.
- Frames with no objects are kept as background-only examples.

Class numbering: `cls` is 0-based and follows `data.yaml`, as in the submission file. The model uses `cls + 1`, because torchvision reserves 0 for background.

In [ ]:
names = yaml.safe_load(open(ROOT / 'train' / 'data.yaml'))['names']
names = [names[k] for k in sorted(names)] if isinstance(names, dict) else names
NUM_CLASSES = len(names)


class CocoFrames(Dataset):
    def __init__(self, split, limit=None, augment=False):
        self.coco = COCO(str(ROOT / split / 'annotations_coco.json'))
        self.augment = augment

        # Remove annotations with an unlisted category or zero size, for training and evaluation alike
        anns = self.coco.dataset['annotations']
        good = [a for a in anns if a['category_id'] in self.coco.cats and a['bbox'][2] > 0 and a['bbox'][3] > 0]
        for a in good:
            a.setdefault('iscrowd', 0)
            a.setdefault('area', a['bbox'][2] * a['bbox'][3])
        self.coco.dataset['annotations'] = good
        self.coco.createIndex()

        # Images are in <split>/ or <split>/images/. Keep only frames that exist on disk.
        for self.folder in [ROOT / split, ROOT / split / 'images']:
            self.ids = [i for i, im in sorted(self.coco.imgs.items()) if (self.folder / im['file_name']).exists()]
            if self.ids:
                break
        assert self.ids, f'No images from the {split} json were found on disk'
        total = len(self.ids)
        if limit and limit < total:  # evenly spaced, so every clip stays represented
            self.ids = [self.ids[k] for k in np.linspace(0, total - 1, limit).astype(int)]

        # COCO category id -> 0-based class index from data.yaml
        cats = sorted(self.coco.cats.values(), key=lambda c: c['id'])
        if all(c['name'] in names for c in cats):
            self.cat2cls = {c['id']: names.index(c['name']) for c in cats}
        else:
            print('WARNING: json category names differ from data.yaml, matching them by order')
            self.cat2cls = {c['id']: k for k, c in enumerate(cats)}
        print(f'{split}: using {len(self.ids):,} frames ({len(self.coco.imgs):,} in json, {total:,} on disk), '
              f'dropped {len(anns) - len(good):,} bad annotations')

    def __len__(self):
        return len(self.ids)

    def boxes_and_labels(self, image_id):
        boxes, labels = [], []
        for a in self.coco.imgToAnns[image_id]:
            x, y, w, h = a['bbox']
            if not a['iscrowd']:
                boxes.append([x, y, x + w, y + h])
                labels.append(self.cat2cls[a['category_id']] + 1)
        return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4), torch.tensor(labels, dtype=torch.int64)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        image = TF.to_tensor(Image.open(self.folder / self.coco.imgs[image_id]['file_name']).convert('RGB'))
        boxes, labels = self.boxes_and_labels(image_id)
        if self.augment and torch.rand(1) < 0.5:  # horizontal flip
            image = image.flip(-1)
            boxes[:, [0, 2]] = image.shape[-1] - boxes[:, [2, 0]]
        return image, {'boxes': boxes, 'labels': labels, 'image_id': image_id}


class TestFrames(Dataset):
    def __init__(self):
        self.paths = sorted((ROOT / 'test' / 'images').rglob('*.jpg'))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]  # .../clip_001/0000001.jpg -> clip_001_0000001
        return TF.to_tensor(Image.open(p).convert('RGB')), {'image_id': f'{p.parent.name}_{p.stem}'}


def collate(batch):
    return tuple(zip(*batch))


train_ds = CocoFrames('train', augment=True)
val_ds = CocoFrames('val')
print(NUM_CLASSES, 'classes:', names)

In [ ]:
# Check the data by eye: the boxes should sit on the objects
image, target = train_ds[0]
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(image.permute(1, 2, 0))
for x1, y1, x2, y2 in target['boxes'].tolist():
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', linewidth=1))
ax.axis('off')
plt.show()

## 3. Model and training setup

| Item | Choice | Why |
|---|---|---|
| Model | torchvision Faster R-CNN, ResNet-50 FPN v2, COCO-pretrained | Standard two-stage baseline, and the FPN gives high-resolution features for small objects |
| Classifier head | Replaced to output the dataset's classes plus background | COCO classes don't match this dataset |
| Anchors | Sizes 16, 32, 64, 128, 256, ratios 0.5, 1, 2 | Default sizes start at 32 px, which suits tiny objects poorly |
| Input size | Short side 1024, long side at most 1824 | Keeps small objects large enough to detect |
| Optimizer | SGD, momentum 0.9, weight decay 1e-4, lr 0.005, batch 2 | Common fine-tuning setup that fits Colab GPU memory |
| Schedule | One-cycle: short warmup, then cosine decay | Works well for short training runs |
| Other | Mixed precision, gradient clipping at 10, horizontal flip | Faster training and stable losses |
| Data | The preprocessed frame subsets of train and val | Neighbouring video frames are nearly identical |

In [ ]:
def build_model(**params):
    model = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=MIN_SIZE, max_size=MAX_SIZE,
        box_detections_per_img=MAX_DET, box_score_thresh=SCORE_THRESH,
        **params,
    )
    # Smaller anchors. Still 3 per location, so the pretrained RPN weights fit.
    model.rpn.anchor_generator = AnchorGenerator(((16,), (32,), (64,), (128,), (256,)), ((0.5, 1.0, 2.0),) * 5)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)  # +1 for background
    return model.to(DEVICE)


def train_model(name, epochs, dataset, **params):
    """Trains and saves after every epoch. An existing checkpoint is continued, or used as is if finished.
    Delete the file in OUT to retrain the same name from scratch."""
    torch.manual_seed(0)
    model = build_model(**params)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=collate, drop_last=True)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                                lr=LR, momentum=0.9, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=epochs * len(loader),
                                                    pct_start=0.1, cycle_momentum=False)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE == 'cuda')

    path = OUT / f'{name}_{epochs}ep.pt'
    start = 0
    if path.exists():
        state = torch.load(path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state['model'])
        start = state['epoch']
        if start < epochs:
            optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler'])
            scaler.load_state_dict(state['scaler'])
        print(f'{name}: loaded checkpoint, {start}/{epochs} epochs done')

    for epoch in range(start, epochs):
        model.train()
        total = 0.0
        for images, targets in tqdm(loader, desc=f'{name} epoch {epoch + 1}/{epochs}'):
            images = [im.to(DEVICE) for im in images]
            targets = [{k: t[k].to(DEVICE) for k in ('boxes', 'labels')} for t in targets]
            with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
                loss = sum(model(images, targets).values())
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += loss.item()
        print(f'{name} epoch {epoch + 1}: mean loss {total / len(loader):.4f}')

        state = {'model': model.state_dict(), 'epoch': epoch + 1}
        if epoch + 1 < epochs:  # finished runs don't need the larger resume state
            state.update(optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(), scaler=scaler.state_dict())
        temp = path.with_suffix('.tmp')
        torch.save(state, temp)
        temp.replace(path)  # a disconnect during saving can't corrupt the checkpoint
    return model

## 4. Prediction and evaluation

`predict` returns `{image_id: (cls, scores, boxes)}` with 0-based classes and `x1, y1, x2, y2` boxes. Evaluation and the submission both use this format.

`pycocotools` counts up to 300 detections per image here, not its default of 100, because these scenes are dense.

In [ ]:
@torch.no_grad()
def predict(model, dataset):
    model.eval()
    loader = DataLoader(dataset, batch_size=4, num_workers=2, collate_fn=collate)
    results = {}
    for images, targets in tqdm(loader, desc='predict'):
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
            outputs = model([im.to(DEVICE) for im in images])
        for out, t in zip(outputs, targets):
            results[t['image_id']] = ((out['labels'] - 1).cpu().numpy(),
                                      out['scores'].float().cpu().numpy(),
                                      out['boxes'].float().cpu().numpy())
    return results


def coco_eval(results, dataset):
    cls2cat = {k: cat for cat, k in dataset.cat2cls.items()}
    detections = [
        {'image_id': image_id, 'category_id': cls2cat[int(c)], 'score': float(s),
         'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]}
        for image_id, (cls, scores, boxes) in results.items()
        for c, s, (x1, y1, x2, y2) in zip(cls, scores, boxes)
    ]
    assert detections, 'The model produced no detections'
    ev = COCOeval(dataset.coco, dataset.coco.loadRes(detections), 'bbox')
    ev.params.imgIds = dataset.ids  # only the frames in the subset
    ev.params.maxDets = [1, 10, MAX_DET]
    ev.evaluate()
    ev.accumulate()

    precision = ev.eval['precision']  # [IoU threshold, recall, class, area range, max detections]
    def mean_ap(p):
        return float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    metrics = {'mAP': mean_ap(precision[:, :, :, 0, -1]),
               'AP50': mean_ap(precision[0, :, :, 0, -1]),
               'AP_small': mean_ap(precision[:, :, :, 1, -1])}
    per_class = {names[dataset.cat2cls[cat]]: mean_ap(precision[:, :, k, 0, -1]) for k, cat in enumerate(ev.params.catIds)}
    return metrics, per_class


@torch.no_grad()
def inference_ms(model, dataset, n=100):
    """Milliseconds per image for the model itself, one image at a time, without file loading."""
    model.eval()
    images = [dataset[i][0].to(DEVICE) for i in range(min(n, len(dataset)))]
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for image in images:
        model([image])
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start) / len(images)

## 5. Parameter study

Two parameters are compared, with everything else fixed:

1. **Number of RPN proposals:** how many candidate regions reach the second stage.
2. **RoI positive fraction:** the share of positive examples among the regions sampled to train the second stage.

These runs are short and use part of the data, so compare them with each other, not with the final model.

In [ ]:
EXPERIMENTS = {
    'baseline':   {},  # torchvision defaults: 2000 train / 1000 test proposals, 25% positive RoIs
    'rpn_low':    {'rpn_post_nms_top_n_train': 500,  'rpn_post_nms_top_n_test': 250},
    'rpn_high':   {'rpn_post_nms_top_n_train': 4000, 'rpn_post_nms_top_n_test': 2000},
    'roi_pos_50': {'box_positive_fraction': 0.5},
}

exp_train = CocoFrames('train', limit=EXP_TRAIN_IMAGES, augment=True)
exp_val = CocoFrames('val', limit=EXP_VAL_IMAGES)

rows = []
for name, params in EXPERIMENTS.items():
    model = train_model(name, EXP_EPOCHS, exp_train, **params)
    metrics, _ = coco_eval(predict(model, exp_val), exp_val)
    rows.append({'run': name, **metrics, 'ms_per_image': inference_ms(model, exp_val)})
    del model
    torch.cuda.empty_cache()

experiments = pd.DataFrame(rows).set_index('run').round(4)
experiments.to_csv(OUT / 'experiments.csv')
best_run = experiments['mAP'].idxmax()
print('Best run:', best_run)
experiments

**Write your findings here, using the table above.**

- RPN proposals: *did more proposals raise mAP or AP_small, and how much slower was inference?*
- RoI positive fraction: *did it change mAP? Compare AP50 with mAP to see whether classification or box accuracy was affected.*

## 6. Final model and validation results

In [ ]:
model = train_model('final', EPOCHS, train_ds, **EXPERIMENTS[best_run])

val_results = predict(model, val_ds)
metrics, per_class = coco_eval(val_results, val_ds)
ms = inference_ms(model, val_ds)

report = pd.Series({**metrics, 'ms_per_image': ms, 'FPS': 1000 / ms}).round(4)
report.to_csv(OUT / 'final_metrics.csv')
print('Parameters from the study:', EXPERIMENTS[best_run] or 'defaults')
print(report, '\n')
print(pd.Series(per_class, name='AP per class').sort_values().round(4))

### What each stage contributes

**Region proposal stage (RPN).** It looks at every location of the FPN feature maps and scores each anchor as object or background, without deciding the class. It also adjusts the anchor boxes and keeps the top few hundred or thousand after NMS. This reduces a very large number of possible boxes to a short list, so its job is mainly **recall**: an object it misses can't be recovered later.

**Second stage (RoI head).** It takes features for each proposal with RoI Align, assigns a class or background, and refines the box. Because it sees features aligned to each candidate and is trained on a balanced sample of regions, it improves **classification and localization precision** and removes many false proposals.

## 7. Failure analysis

In [ ]:
def find_errors(results, dataset, conf=0.5, iou_thr=0.5):
    """For each frame, sorts confident predictions and ground truth into three kinds of error."""
    errors = []
    for idx, image_id in enumerate(dataset.ids):
        cls, scores, boxes = results[image_id]
        keep = scores >= conf
        pred_boxes, pred_cls = torch.as_tensor(boxes[keep]).reshape(-1, 4), torch.as_tensor(cls[keep])
        gt_boxes, gt_labels = dataset.boxes_and_labels(image_id)
        gt_cls = gt_labels - 1

        overlap = box_iou(gt_boxes, pred_boxes) >= iou_thr
        correct = overlap & (gt_cls[:, None] == pred_cls[None, :])
        errors.append({
            'idx': idx,
            'missed': gt_boxes[~overlap.any(1)],                  # no prediction on this object
            'confused': pred_boxes[overlap.any(0) & ~correct.any(0)],  # right place, wrong class
            'false_pos': pred_boxes[~overlap.any(0)],             # prediction on nothing
            'gt_sizes': (gt_boxes[:, 2:] - gt_boxes[:, :2]).prod(1).sqrt(),
            'missed_mask': ~overlap.any(1),
        })
    return errors


def show_frame(error, kind, dataset):
    image = dataset[error['idx']][0]
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(image.permute(1, 2, 0))
    for x1, y1, x2, y2 in error[kind].tolist():
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='red', linewidth=1.5))
    file_name = dataset.coco.imgs[dataset.ids[error['idx']]]['file_name']
    ax.set_title(f"{kind}: {len(error[kind])} boxes in {file_name}")
    ax.axis('off')
    plt.show()


errors = find_errors(val_results, val_ds)

# How often objects are missed, by size
sizes = torch.cat([e['gt_sizes'] for e in errors])
missed = torch.cat([e['missed_mask'] for e in errors])
print('Share of objects missed at confidence 0.5, by size (square root of box area):')
for low, high in [(0, 16), (16, 32), (32, 96), (96, 1e9)]:
    in_bin = (sizes >= low) & (sizes < high)
    if in_bin.any():
        print(f'  {low:>3}-{high if high < 1e9 else "up":>3} px: {missed[in_bin].float().mean():.1%} of {int(in_bin.sum()):,}')

# The frame with the most errors of each kind
for kind in ['missed', 'confused', 'false_pos']:
    show_frame(max(errors, key=lambda e: len(e[kind])), kind, val_ds)

**Write up three failure cases from the frames above.** Describe what you can actually see, and treat each cause as a likely explanation.

| Case | What you see | Possible causes to consider |
|---|---|---|
| Missed objects | *fill in* | Very few pixels per object, crowding, proposals removed by NMS in dense groups, blur or darkness |
| Wrong class | *fill in* | Similar-looking classes from above (for example car and van), too little detail at small sizes, few training examples of the class |
| False positives | *fill in* | Background that resembles objects, or real objects that weren't annotated, such as those in ignored regions |

## 8. Kaggle submission

In [ ]:
def make_submission(model):
    """Predicts every test image and returns {image_id: (cls, scores, boxes)}. Nothing is written to disk."""
    return predict(model, TestFrames())  # always every test image, never a subset


def save_submission(results, path):
    """Writes predictions as: image_id,PredictionString with "cls score x y w h" for each box.
    Every test image gets a row, and a frame with no boxes gets an empty PredictionString."""
    def to_text(cls, scores, boxes):
        return ' '.join(f'{c} {s:.4f} {x1:.1f} {y1:.1f} {x2 - x1:.1f} {y2 - y1:.1f}'
                        for c, s, (x1, y1, x2, y2) in zip(cls, scores, boxes))

    submission = pd.DataFrame({'image_id': sorted(results)})
    submission['PredictionString'] = submission['image_id'].map(lambda image_id: to_text(*results[image_id]))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    print(f'Saved {path} with {len(submission):,} rows')
    return submission

In [ ]:
# Load the final model from Drive, then keep the predictions in memory
model = (m := build_model(), m.load_state_dict(torch.load(OUT / 'final_6ep.pt', map_location=DEVICE, weights_only=False)['model']))[0].eval()
test_results = make_submission(model)
print(f'Predicted {len(test_results):,} test images')

In [ ]:
# If saving fails, fix the path and re-run only this cell: test_results stays in memory
submission = save_submission(test_results, OUT / 'submission_faster_rcnn.csv')
submission.head()